In [11]:
import pandas as pd
import torch
import numpy as np
import gc
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback  # [추가된 부분] 1. 조기 종료 콜백 모듈 임포트
)
from torch.optim import AdamW 

# ---------------------------------------------
# 0. 메모리 초기화
# ---------------------------------------------
gc.collect()
torch.cuda.empty_cache()

# ---------------------------------------------
# 1. 데이터 로드 및 전처리
# ---------------------------------------------
train_combined_path = './data/final_train_data2.csv'
df = pd.read_csv(train_combined_path)
df = df.dropna(subset=['conversation', 'label'])

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# ---------------------------------------------
# 2. 모델 및 토크나이저 로드
# ---------------------------------------------
MAX_LENGTH = 256
BATCH_SIZE = 32
# [수정된 부분] 2. 조기 종료가 있으므로 최대 에포크를 넉넉하게 10으로 늘립니다. 
# 어차피 성능이 안 오르면 중간에 알아서 멈춥니다!
EPOCHS = 10          
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.001    

MODEL_NAME = "beomi/KcELECTRA-base-v2022" 
num_labels = len(df['label'].unique()) 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

# ---------------------------------------------
# 3. Custom Dataset 클래스
# ---------------------------------------------
class KcElectraDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = KcElectraDataset(train_df['conversation'].values, train_df['label'].values, tokenizer, MAX_LENGTH)
val_dataset = KcElectraDataset(val_df['conversation'].values, val_df['label'].values, tokenizer, MAX_LENGTH)

# ---------------------------------------------
# 4. 평가지표 계산 함수
# ---------------------------------------------
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    
    return {
        'accuracy': acc,
        'f1_macro': f1
    }

# ---------------------------------------------
# 5. Custom Optimizer 및 Trainer 셋팅
# ---------------------------------------------
no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_parameters = [
    {
        "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
        "weight_decay": WEIGHT_DECAY,
    },
    {
        "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
    },
]

custom_optimizer = AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE, eps=1e-5)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=EPOCHS,               
    per_device_train_batch_size=BATCH_SIZE,        
    per_device_eval_batch_size=BATCH_SIZE,
    max_grad_norm=1.0,                     
    warmup_ratio=0.1,                      
    lr_scheduler_type="linear",            
    seed=42,                               
    eval_strategy="epoch",                 
    save_strategy="epoch",                 
    logging_steps=50,                      
    report_to="none",
    
    # [추가된 부분] 3. 조기 종료 및 최고 모델 저장 설정
    load_best_model_at_end=True,        # 학습 종료 시 가장 성능이 좋았던 가중치로 덮어씌움
    metric_for_best_model="f1_macro",   # 어떤 점수를 기준으로 최고 모델을 뽑을지 결정 (대회 기준인 f1_macro 사용)
    greater_is_better=True,             # F1 점수는 높을수록 좋으므로 True
    save_total_limit=2                  # 디스크 용량 관리를 위해 가장 최근 모델/최고 모델 2개만 남기고 삭제
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    optimizers=(custom_optimizer, None),
    # [추가된 부분] 4. 조기 종료 조건 부여 (patience=2: 2번의 에포크 동안 f1_macro가 안 오르면 종료)
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] 
)

print("▶ [1/2] 모델 학습을 시작합니다...\n")
trainer.train()

# ==========================================
# 6. 최종 평가 및 리포트 출력 (Evaluation)
# ==========================================
print("\n▶ [2/2] 최종 모델 성능 평가를 진행합니다...")

# [추가된 부분] 5. 학습이 일찍 멈추더라도 가장 좋았던 점수(best_model)를 기준으로 평가가 진행됩니다.
eval_results = trainer.evaluate()

print("-" * 50)
print("🎯 [최종 검증 세트(Validation Set) 평가 결과]")
print(f" - Loss (손실): {eval_results['eval_loss']:.4f}")
print(f" - Accuracy (정확도): {eval_results['eval_accuracy'] * 100:.2f}%")
print(f" - F1 Score (Macro): {eval_results['eval_f1_macro']:.4f}")
print("-" * 50)

print("\n▶ 검증 데이터셋 상세 예측 중...")
output = trainer.predict(val_dataset)
preds = np.argmax(output.predictions, axis=-1)

target_names = ['협박 대화', '갈취 대화', '직장 내 괴롭힘 대화', '기타 괴롭힘 대화', '일반 대화']

print("\n" + "="*60)
print("🎯 [KcELECTRA 상세 성능 리포트]")
print("="*60)
print(classification_report(val_df['label'].values, preds, target_names=target_names, zero_division=0))
print("="*60) 

save_directory = "./best_kcelectra_model13"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"\n✅ 학습 완료! 조기 종료가 적용된 최고 성능의 모델이 '{save_directory}' 폴더에 저장되었습니다.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base-v2022
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.

▶ [1/2] 모델 학습을 시작합니다...



Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.354995,0.733250,0.883317,0.873651
2,0.352151,0.252295,0.929605,0.924329
3,0.215889,0.301921,0.909354,0.902168
4,0.119226,0.219971,0.939248,0.933999
5,0.100787,0.257869,0.933462,0.928546
6,0.034419,0.277993,0.937319,0.932482


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.034419,0.219971,6,0.939248,0.933999


--------------------------------------------------
🎯 [최종 검증 세트(Validation Set) 평가 결과]
 - Loss (손실): 0.2200
 - Accuracy (정확도): 93.92%
 - F1 Score (Macro): 0.9340
--------------------------------------------------

▶ 검증 데이터셋 상세 예측 중...



🎯 [KcELECTRA 상세 성능 리포트]
              precision    recall  f1-score   support

       협박 대화       0.90      0.90      0.90       178
       갈취 대화       0.91      0.94      0.92       195
 직장 내 괴롭힘 대화       0.95      0.98      0.96       194
   기타 괴롭힘 대화       0.92      0.86      0.89       202
       일반 대화       1.00      1.00      1.00       268

    accuracy                           0.94      1037
   macro avg       0.93      0.93      0.93      1037
weighted avg       0.94      0.94      0.94      1037



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ 학습 완료! 조기 종료가 적용된 최고 성능의 모델이 './best_kcelectra_model13' 폴더에 저장되었습니다.


In [12]:
import pandas as pd
from transformers import pipeline

# 1. 파일 로드 및 모델 준비
test_df = pd.read_csv("data/test.csv")
submission_df = pd.read_csv("data/submission_default.csv")

classifier = pipeline(
    "text-classification", 
    model=trainer.model, 
    tokenizer=tokenizer, 
    device=0 
)

# 2. 추론 실행 
print(f"총 {len(test_df)}건 추론 중...")
results = classifier(test_df['conversation'].tolist(), batch_size=16)

# 3. 결과 정리 (핵심 변경 부분!)
# res['label']은 'LABEL_0' 형태로 나오므로, 문자열에서 숫자만 추출(int)합니다.
# 예: 'LABEL_0' -> 0
predictions = [int(res['label'].split('_')[-1]) for res in results]

# 4. submission 양식에 채우기
submission_df['class'] = predictions

# 5. 저장 (숫자만 들어가므로 인코딩 옵션은 빼도 무방합니다)
output_path = "data/submission_KcELECTRA13.csv"
submission_df.to_csv(output_path, index=False)

print("-" * 50)
print(f"✅ 제출 파일 생성 완료: {output_path}")

# 6. 결과 확인
print(submission_df.head(10))

총 500건 추론 중...
--------------------------------------------------
✅ 제출 파일 생성 완료: data/submission_KcELECTRA13.csv
     idx  class
0  t_000      1
1  t_001      2
2  t_002      2
3  t_003      4
4  t_004      3
5  t_005      0
6  t_006      0
7  t_007      1
8  t_008      3
9  t_009      1
